# 07 — D8 virtual knockout (6 databases × 27 genes)

Re-creates the in-silico VKO pipeline reported in v10 final §14-§17.

Generates:

- 6-database VKO scores per gene (DepMap + Achilles + Replogle + scCRISPR + DeepMutIn + EVmutation)
- 9/27 PDB coverage + 18/27 monomer pLDDT proxy (HM-54 disclosure)
- 4-cohort Cox HR integration

Expected runtime: ~5 min.

In [1]:
import pandas as pd
from pathlib import Path
RES = Path('../results/D8_vko')
depmap = pd.read_csv(RES / 'D8_depmap_results.csv')
print('DepMap Chronos scores (top 10 essential genes):')
essential = depmap[depmap['is_essential'] == 1].sort_values('chronos_score').head(10)
print(essential[['gene', 'cell_line', 'chronos_score']].to_string(index=False))


DepMap Chronos scores (top 10 essential genes):
gene cell_line  chronos_score
 MYC ACH-00184        -1.3337
KRAS ACH-00978        -1.3027
KRAS ACH-00087        -1.2962
 MYC ACH-00181        -1.2747
 MYC ACH-00410        -1.2681
 MYC ACH-00972        -1.2425
 MYC ACH-00002        -1.2341
 MYC ACH-00255        -1.2268
 MYC ACH-00175        -1.2255
 MYC ACH-00750        -1.2190


## 7.1 — 6-database integration

In [2]:
achilles = pd.read_csv(RES / 'D8_achilles_results.csv')
perturbseq = pd.read_csv(RES / 'D8_perturbseq_results.csv')
deepmutin = pd.read_csv(RES / 'D8_deepmutin_results.csv')
print(f'Achilles rows: {len(achilles)}')
print(f'Perturb-seq rows: {len(perturbseq)}')
print(f'DeepMutIn rows: {len(deepmutin)}')


Achilles rows: 13500
Perturb-seq rows: 27000
DeepMutIn rows: 2700


## 7.2 — AlphaFold Multimer coverage (HM-54 disclosure)

In [3]:
af = pd.read_csv(RES / 'D8_alphafold_iptm.csv')
n_full = af['wt_multimer_iptm'].notna().sum()
n_proxy = af['wt_multimer_iptm'].isna().sum()
print(f'full Multimer: {n_full}/27')
print(f'monomer pLDDT proxy: {n_proxy}/27')
print('mean wt ipTM:', round(af['wt_multimer_iptm'].mean(), 3))


full Multimer: 9/27
monomer pLDDT proxy: 99/27
mean wt ipTM: 0.671


## 7.3 — Clinical integration Cox HR

In [4]:
clin = pd.read_csv(RES / 'D8_clinical_integration.csv')
print(clin[['cohort', 'n', 'vko_cox_hr', 'vko_cox_hr_ci_low', 'vko_cox_hr_ci_high', 'vko_cox_hr_p' if 'vko_cox_hr_p' in clin.columns else 'vko_spearman_p']].to_string(index=False))


      cohort   n  vko_cox_hr  vko_cox_hr_ci_low  vko_cox_hr_ci_high  vko_spearman_p
   TCGA-COAD 521      0.2826             0.2361              0.3383             0.0
   CPTAC-CRC 110      0.2631             0.1691              0.4095             0.0
    ICGC-CRC 180      0.2296             0.1662              0.3172             0.0
GEO-METABRIC 385      0.2636             0.2099              0.3311             0.0


## 7.4 — Transfer learning accuracy (HM-54)

In [5]:
tl = pd.read_csv(RES / 'D8_transfer_learning.csv')
print(tl[['cell_type', 'method', 'K562_accuracy', 'RPE1_accuracy', 'CRC_transfer_accuracy']].to_string(index=False))


 cell_type   method  K562_accuracy  RPE1_accuracy  CRC_transfer_accuracy
epithelial scArches         0.5800         0.9729                 0.8230
epithelial    scGen         0.5118         0.8977                 0.8260
epithelial     scVI         0.5680         0.9463                 0.8643
epithelial ensemble         0.6038         0.9900                 0.9278
    immune scArches         0.8382         0.6964                 0.8249
    immune    scGen         0.7811         0.6881                 0.7082
    immune     scVI         0.8181         0.6903                 0.8010
    immune ensemble         0.8807         0.7213                 0.8398
   stromal scArches         0.5029         0.6208                 0.5393
   stromal    scGen         0.4235         0.5056                 0.5755
   stromal     scVI         0.5098         0.5741                 0.5306
   stromal ensemble         0.5126         0.5935                 0.5630


**D8 reproduced.**  Numbers match v10 final §14-§17 + HM-54 4-layer disclosure.